# Week 6 Lab 5.1: Structured LLM Evaluation and Experiment Tracking

**Scenario:** Cordwell Home and Hardware, a fictional big-box home improvement retailer
**Duration:** about 2.5 hours core, plus optional stretch goals
**Stack:** TruLens 2.12, MLflow 3.15, sacrebleu, rouge-score, scikit-learn (isolated venv, see the README)

You are on the team that ships Cordwell's customer-facing product assistant. Before the next release, you need to answer two questions every production LLM team faces:

1. **Is the new version actually better?** You will build an evaluation harness that scores the assistant with traditional metrics (accuracy, F1, BLEU, ROUGE) and with structured RAG metrics (groundedness, answer relevance, context relevance).
2. **Can you prove it later?** You will log every experiment to MLflow so that params, metrics, and per-example predictions are tracked, versioned, and comparable side by side.

## Learning objectives

By the end of this lab you will be able to:

1. Implement token-level F1, BLEU, and ROUGE scoring for free-form LLM output, including the argument-order traps in `rouge_score` and `sacrebleu`.
2. Build an evaluation harness that runs an LLM app over a labeled dataset and returns a metrics dictionary plus per-example predictions.
3. Log parameters, metrics, and prediction artifacts to MLflow and compare app variants in the MLflow UI.
4. Instrument a RAG app with TruLens spans and write metric functions for the RAG triad: groundedness, answer relevance, and context relevance.
5. Join TruLens leaderboard scores into MLflow runs so one tracking system holds the whole picture.

## How this lab works

- Cells marked **GIVEN** are plumbing. Run them and read the short callouts, but your active time belongs to the **TASK** cells.
- Each task has a **Worked target output** block just above it showing exactly what a correct implementation produces for a real input. Code toward that target.
- A soft check harness verifies your work as you go. On a fresh Run All the notebook opens with most checks in TODO status and zero crashes. That is expected.
- Two hint files ship with the lab: `HINTS.md` (three escalating levels per task) and `HINTS_DETAILED.md` (the working core with line-by-line commentary). Pick one tier per task; reading both wastes time.

## 0. Environment and configuration (GIVEN)

### Backends

All LLM calls go through one `call_llm` function with three modes, selected by the `BACKEND_MODE` environment variable:

| Mode | What it does |
|---|---|
| `offline` (default) | A deterministic stub answers from the retrieved context. No servers, no network. Every number in this notebook is reproducible in this mode, and the check harness expects it. |
| `lmstudio` | Real model through LM Studio's OpenAI-compatible server on port 1234. |
| `ollama` | Real model through Ollama's OpenAI-compatible endpoint on port 11434. |

The two live backends share one client code path and differ only by base URL. The model name lives in a config variable defaulting to the cohort Gemma tag. If a live server is down, the client raises `openai.APIConnectionError`. There is no silent fallback.

### MLflow

Experiment tracking works in two modes, selected by the `MLFLOW_TRACKING_URI` environment variable:

- **Local file mode (default):** runs are stored in a SQLite file next to this notebook. Zero dependencies. View them later with `mlflow ui --backend-store-uri sqlite:///mlflow_local.db --port 5001`.
- **Docker server mode (classroom):** a tracking server in Docker on port 5000. Setup commands are in the README. Export `MLFLOW_TRACKING_URI=http://localhost:5000` before starting Jupyter and the exact same logging code talks to the server instead.

In [ ]:
%pip install -r requirements.txt

In [ ]:
import os
import re
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd

BACKEND_MODE = os.environ.get("BACKEND_MODE", "offline").lower()
MODEL_NAME = os.environ.get("CORDWELL_MODEL_NAME", "gemma4")

BACKEND_URLS = {
    "lmstudio": "http://localhost:1234/v1",
    "ollama": "http://localhost:11434/v1",
}

if BACKEND_MODE not in ("offline", "lmstudio", "ollama"):
    raise ValueError(
        f"BACKEND_MODE must be offline, lmstudio, or ollama, got {BACKEND_MODE!r}"
    )

LAB_DIR = Path(".").resolve()
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
EXPERIMENT_NAME = "week6_lab51_cordwell_eval"

if BACKEND_MODE in BACKEND_URLS:
    from openai import OpenAI

    llm_client = OpenAI(base_url=BACKEND_URLS[BACKEND_MODE], api_key="not-needed")
else:
    llm_client = None

print(f"Backend mode : {BACKEND_MODE}")
print(f"Model name   : {MODEL_NAME}")
print(f"MLflow URI   : {MLFLOW_TRACKING_URI}")

In [ ]:
# Soft check harness (GIVEN). Each check ends in one of four states:
#   PASS  the assertion held
#   FAIL  the assertion did not hold, or the code raised an unexpected error
#   TODO  the code raised NotImplementedError, meaning the task is not started
#   SKIP  the check only applies in offline mode and BACKEND_MODE is live

CHECK_RESULTS = {}


class SkipCheck(Exception):
    pass


def require_offline():
    if BACKEND_MODE != "offline":
        raise SkipCheck("only checked in offline mode")


def check(check_id: str, description: str, fn):
    try:
        fn()
        status, detail = "PASS", ""
    except NotImplementedError:
        status, detail = "TODO", "not implemented yet"
    except SkipCheck as err:
        status, detail = "SKIP", str(err)
    except AssertionError as err:
        status, detail = "FAIL", str(err)
    except Exception as err:
        status, detail = "FAIL", f"{type(err).__name__}: {err}"
    CHECK_RESULTS[check_id] = (description, status, detail)
    marker = {"PASS": "PASS ", "FAIL": "FAIL ", "TODO": "TODO ", "SKIP": "SKIP "}[status]
    line = f"[{marker}] {check_id}: {description}"
    if detail:
        line += f" ({detail})"
    print(line)


def close_to(actual, expected, tolerance=1e-9):
    assert abs(actual - expected) <= tolerance, f"expected {expected}, got {actual}"


def run_summary():
    counts = {"PASS": 0, "FAIL": 0, "TODO": 0, "SKIP": 0}
    for _, status, _ in CHECK_RESULTS.values():
        counts[status] += 1
    total = len(CHECK_RESULTS)
    print(f"Checks: {counts['PASS']} PASS, {counts['FAIL']} FAIL, "
          f"{counts['TODO']} TODO, {counts['SKIP']} SKIP out of {total}")
    for check_id, (description, status, detail) in CHECK_RESULTS.items():
        if status != "PASS":
            print(f"  [{status}] {check_id}: {description}")


print("Check harness ready.")

## 1. The Cordwell corpus (GIVEN)

The assistant answers questions over a synthetic knowledge base: 300 short documents across 10 product categories and 6 document types (product guides, policy FAQs, installation guides, troubleshooting notes, promotions, and customer reviews). Reviews carry a hidden gold sentiment label we will use later.

Everything is generated from a fixed seed, so every student's corpus is byte-identical. That is what makes the check numbers in this notebook exact rather than approximate.

In [ ]:
CATEGORIES = [
    "laminate flooring",
    "vinyl plank flooring",
    "ceramic tile",
    "interior paint",
    "exterior paint",
    "LED lighting",
    "kitchen cabinets",
    "bathroom vanities",
    "power tools",
    "garden and outdoor",
]

DOC_TYPES = [
    "product_guide",
    "policy_faq",
    "installation_guide",
    "troubleshooting",
    "promotion",
    "customer_review",
]

WARRANTY_TERMS = ["1-year", "2-year", "3-year", "5-year", "10-year"]
RETURN_WINDOWS = ["30 days", "60 days", "90 days"]
FINISH_LEVELS = ["matte", "eggshell", "satin", "semi-gloss"]
TRAFFIC_LEVELS = ["light", "medium", "heavy"]

REVIEW_PHRASES = {
    "positive": [
        "loved how easy the install was",
        "customer service was fantastic",
        "quality felt better than expected",
        "installer arrived on time and cleaned up thoroughly",
        "the color matched the sample perfectly",
    ],
    "negative": [
        "installation was delayed and communication was poor",
        "finish scratched more easily than expected",
        "return process felt confusing and slow",
        "instructions were unclear and missing steps",
        "delivery arrived late and boxes were damaged",
    ],
    "neutral": [
        "product quality was fine, nothing special",
        "store was crowded but staff eventually helped",
        "color was close enough to what we expected",
        "installer did the job, but scheduling took a few calls",
        "overall experience was acceptable but not memorable",
    ],
}


def build_corpus(docs_per_category: int = 30) -> pd.DataFrame:
    """Generate the deterministic Cordwell corpus. Seeded inside so a
    re-run of this cell always rebuilds the identical corpus."""
    random.seed(42)
    rows = []
    doc_id = 0
    for category in CATEGORIES:
        for _ in range(docs_per_category):
            d_type = random.choice(DOC_TYPES)
            warranty = random.choice(WARRANTY_TERMS)
            return_window = random.choice(RETURN_WINDOWS)
            finish = random.choice(FINISH_LEVELS)
            traffic = random.choice(TRAFFIC_LEVELS)

            if d_type == "product_guide":
                text = (
                    f"Cordwell Home and Hardware carries several {category} options designed for {traffic} traffic areas. "
                    f"Most {category} products come with a {warranty} limited residential warranty when "
                    f"installed according to the manufacturer guidelines. Customers should check the "
                    f"subfloor, moisture levels, and acclimation requirements before starting. Many customers "
                    f"prefer a {finish} finish to balance durability and appearance. Stores typically keep "
                    f"popular colors and textures in stock, and special orders may take 7 to 14 days."
                )
            elif d_type == "policy_faq":
                text = (
                    f"The Cordwell return policy for {category} balances flexibility with product quality. "
                    f"Most unopened {category} purchases can be returned within {return_window} with proof of purchase. "
                    f"Cut-to-length and custom-tinted products may not be returnable. For installed {category}, "
                    f"customers should contact the installation support team before removal to avoid damage. "
                    f"Refunds are generally issued to the original payment method."
                )
            elif d_type == "installation_guide":
                text = (
                    f"When installing {category}, Cordwell recommends dry-fitting a small area before committing "
                    f"to full coverage. For {traffic} traffic areas, use the recommended underlayment and follow "
                    f"the expansion gap guidelines listed on the packaging. Surfaces must be clean, flat, and "
                    f"structurally sound. For {finish} finishes, avoid harsh cleaners for the first 7 days after "
                    f"installation. Printed guides are available in store and video tutorials are online."
                )
            elif d_type == "troubleshooting":
                text = (
                    f"Common issues with {category} include minor color variation between batches and surface "
                    f"noise in {traffic} traffic hallways. Color variation can be reduced by mixing pieces from "
                    f"multiple boxes during installation. For squeaks or hollow sounds, verify the subfloor prep "
                    f"and confirm the recommended underlayment was used. The installation support line can review "
                    f"photos and suggest remedies or warranty options."
                )
            elif d_type == "promotion":
                text = (
                    f"During the seasonal Cordwell Savings Event, select {category} products may qualify for "
                    f"bundled discounts. Typical offers include percentage discounts on minimum square footage "
                    f"purchases and free delivery on qualifying orders. Promotions on {category} sometimes combine "
                    f"with installation offers, such as discounted labor on weekday installs. Clearance items and "
                    f"certain premium collections may be excluded."
                )
            else:
                sentiment = random.choice(["positive", "negative", "neutral"])
                phrase = random.choice(REVIEW_PHRASES[sentiment])
                pricing = "fair" if sentiment != "negative" else "higher than I expected"
                again = "would" if sentiment == "positive" else "might"
                staff = (
                    "answered most of my questions"
                    if sentiment != "negative"
                    else "seemed rushed and hard to flag down"
                )
                sched = "smooth" if sentiment == "positive" else "a little bumpy"
                text = (
                    f"I recently purchased {category} from Cordwell Home and Hardware. "
                    f"I {phrase}. The pricing felt {pricing}, and I {again} shop here again for future projects. "
                    f"The store team {staff}. Installation scheduling was {sched}."
                )
                rows.append(
                    {
                        "doc_id": doc_id,
                        "category": category,
                        "doc_type": d_type,
                        "sentiment": sentiment,
                        "text": text,
                    }
                )
                doc_id += 1
                continue

            rows.append(
                {
                    "doc_id": doc_id,
                    "category": category,
                    "doc_type": d_type,
                    "sentiment": None,
                    "text": text,
                }
            )
            doc_id += 1
    return pd.DataFrame(rows)


corpus_df = build_corpus()
print(f"Corpus size: {len(corpus_df)} documents")
print(corpus_df["doc_type"].value_counts())
corpus_df.sample(3, random_state=7)

## 2. The evaluation dataset (GIVEN)

You cannot measure quality without labeled examples. The evaluation set has 60 rows across three task types, each exercising a different kind of metric:

| Task type | Rows | Input | Reference | Metrics later |
|---|---|---|---|---|
| `faq` | 20 | A customer question | A short gold answer | BLEU, ROUGE-L, token F1 |
| `summarization` | 15 | A full installation guide | A gold 2 to 3 sentence summary | BLEU, ROUGE-L, token F1 |
| `sentiment` | 25 | A customer review | A gold label: positive, negative, neutral | Accuracy, macro precision, recall, F1 |

Note the honest weakness baked in: the FAQ and summarization references are template paraphrases, not exact copies of what any model would say. Free-form model output will never match them word for word. That is the point. You are about to see what reference-based metrics look like on realistic LLM output, and later why structured metrics exist.

In [ ]:
def build_eval_dataset(corpus: pd.DataFrame) -> pd.DataFrame:
    """Build the labeled evaluation set from the corpus. Sampling uses
    fixed random_state values, so the result is deterministic."""
    rows = []

    faq_pool = corpus[corpus["doc_type"] == "policy_faq"].sample(n=20, random_state=123)
    for _, row in faq_pool.iterrows():
        cat = row["category"]
        question = f"What is the return policy for {cat} at Cordwell?"
        reference = (
            f"Most unopened {cat} purchases can be returned within the posted return window "
            f"with proof of purchase. Cut-to-length and custom-tinted products may not be "
            f"returnable, and refunds generally go back to the original payment method."
        )
        rows.append(
            {
                "task_type": "faq",
                "category": cat,
                "source_doc_id": row["doc_id"],
                "input_text": question,
                "reference_text": reference,
                "sentiment_label": None,
            }
        )

    summ_pool = corpus[corpus["doc_type"] == "installation_guide"].sample(n=15, random_state=456)
    for _, row in summ_pool.iterrows():
        cat = row["category"]
        reference = (
            f"Before installing {cat}, dry-fit a small area, prepare a clean and flat surface, "
            f"use the recommended underlayment, and follow the expansion gap guidelines. Avoid "
            f"harsh cleaners right after installation and use the printed or online guides."
        )
        rows.append(
            {
                "task_type": "summarization",
                "category": cat,
                "source_doc_id": row["doc_id"],
                "input_text": row["text"],
                "reference_text": reference,
                "sentiment_label": None,
            }
        )

    review_pool = corpus[corpus["doc_type"] == "customer_review"].sample(n=25, random_state=789)
    for _, row in review_pool.iterrows():
        rows.append(
            {
                "task_type": "sentiment",
                "category": row["category"],
                "source_doc_id": row["doc_id"],
                "input_text": row["text"],
                "reference_text": None,
                "sentiment_label": row["sentiment"],
            }
        )

    return pd.DataFrame(rows)


eval_df = build_eval_dataset(corpus_df)
print(f"Evaluation set: {len(eval_df)} rows")
print(eval_df["task_type"].value_counts())
print("Sentiment label distribution:")
print(eval_df[eval_df["task_type"] == "sentiment"]["sentiment_label"].value_counts())
eval_df.head(3)

## 3. The Cordwell RAG app and its backends (GIVEN)

The app under evaluation is deliberately simple, because Week 5 already covered RAG engineering and this week is about measuring, not building:

1. `retrieve_contexts` ranks the corpus against the query with TF-IDF cosine similarity and returns the top k document texts.
2. `build_messages` assembles a chat prompt. Two prompt styles exist: `baseline` and `strong_grounding`, which orders the model to answer only from context and to say it does not know otherwise. These are the A and B of the A-B test you will run.
3. `call_llm` routes the messages to the active backend. In `offline` mode a deterministic stub plays the role of the model.

### How the offline stub thinks

The stub reads the prompt exactly the way a real model would receive it, then answers mechanically:

- For FAQ and summarization prompts it extracts the first retrieved document from the CONTEXT block and returns its first two sentences. An extractive answer like this is perfectly grounded but only sometimes relevant, which will show up in the metrics later.
- Under `strong_grounding` it appends one hedge sentence pointing the customer to a store associate. Watch what that one sentence does to groundedness scores later.
- For sentiment prompts it counts hits from small positive and negative word lists and picks the winner, defaulting to neutral on a tie. Two review phrasings are invisible to those lists on purpose, so the stub misclassifies them. Deterministic imperfection is what makes accuracy and F1 worth computing.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer(stop_words="english")
doc_matrix = vectorizer.fit_transform(corpus_df["text"].tolist())


def retrieve_contexts(query: str, top_k: int = 4) -> list:
    """Return the top_k most similar corpus texts for the query."""
    query_vec = vectorizer.transform([query])
    sims = cosine_similarity(query_vec, doc_matrix).ravel()
    top_idx = np.argsort(sims)[::-1][:top_k]
    return corpus_df.iloc[top_idx]["text"].tolist()


def build_messages(query: str, contexts: list, task_type: str, prompt_style: str) -> list:
    """Assemble the chat messages for one request."""
    context_block = "\n\n---\n\n".join(contexts)
    if prompt_style == "strong_grounding":
        system_msg = (
            "You are the Cordwell Home and Hardware assistant. Base your answer ONLY on the "
            "provided context. If the context does not contain the answer, say you do not know "
            "and suggest talking to a store associate. Be concise and specific."
        )
    else:
        system_msg = (
            "You are the Cordwell Home and Hardware assistant. Use the context when relevant, "
            "and keep answers short and helpful."
        )

    if task_type == "faq":
        user_msg = (
            f"CONTEXT:\n{context_block}\n\nQUESTION: {query}\n\n"
            "Answer the customer question in 2 to 4 sentences."
        )
    elif task_type == "summarization":
        user_msg = (
            f"CONTEXT:\n{context_block}\n\n"
            "TASK: Summarize the key guidance for a customer in 2 to 3 sentences."
        )
    else:  # sentiment
        user_msg = (
            "Classify the overall sentiment of this customer review as positive, negative, "
            f"or neutral. Reply with exactly one word.\n\nREVIEW:\n{query}"
        )
    return [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]


_SENTENCE_SPLIT = re.compile(r"(?<=[.!?])\s+")

POSITIVE_WORDS = ["loved", "fantastic", "better than expected", "on time", "cleaned up"]
NEGATIVE_WORDS = ["delayed", "poor", "scratched", "unclear", "missing", "damaged", "late"]


def offline_stub_response(messages: list) -> str:
    """A deterministic stand-in for a chat model. It reads the same
    prompt a real model would and answers mechanically."""
    user_msg = messages[-1]["content"]
    system_msg = messages[0]["content"]

    if user_msg.startswith("Classify the overall sentiment"):
        review = user_msg.split("REVIEW:", 1)[1].lower()
        pos_hits = sum(1 for w in POSITIVE_WORDS if w in review)
        neg_hits = sum(1 for w in NEGATIVE_WORDS if w in review)
        if pos_hits > neg_hits:
            return "positive"
        if neg_hits > pos_hits:
            return "negative"
        return "neutral"

    context_block = user_msg.split("CONTEXT:", 1)[1]
    for marker in ("\n\nQUESTION:", "\n\nTASK:"):
        if marker in context_block:
            context_block = context_block.split(marker, 1)[0]
            break
    first_context = context_block.strip().split("\n\n---\n\n")[0]
    sentences = _SENTENCE_SPLIT.split(first_context.strip())
    answer = " ".join(sentences[:2])
    if "ONLY on the" in system_msg:
        answer += " If details differ at your store, please check with an associate."
    return answer


def call_llm(messages: list) -> str:
    """Route the chat messages to the active backend."""
    if BACKEND_MODE == "offline":
        return offline_stub_response(messages)
    response = llm_client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        temperature=0.0,
        max_tokens=256,
    )
    return response.choices[0].message.content


def run_cordwell_app(query: str, task_type: str, top_k: int = 4, prompt_style: str = "baseline") -> dict:
    """Run one end-to-end request and return the answer plus the contexts used."""
    if task_type == "sentiment":
        contexts = []
    else:
        contexts = retrieve_contexts(query, top_k=top_k)
    messages = build_messages(query, contexts, task_type, prompt_style)
    answer = call_llm(messages)
    return {"answer": answer, "contexts": contexts}

In [ ]:
# Smoke test (GIVEN). Two calls, one of each flavor.
faq_row = eval_df[eval_df["task_type"] == "faq"].iloc[0]
faq_out = run_cordwell_app(faq_row["input_text"], "faq")
print("FAQ question :", faq_row["input_text"])
print("FAQ answer   :", faq_out["answer"])
print()
sent_row = eval_df[eval_df["task_type"] == "sentiment"].iloc[0]
sent_out = run_cordwell_app(sent_row["input_text"], "sentiment")
print("Review (start):", sent_row["input_text"][:90], "...")
print("Predicted     :", sent_out["answer"])
print("Gold label    :", sent_row["sentiment_label"])

In offline mode the smoke test above already shows a **misclassification**: the first sentiment review is gold `positive` but predicted `neutral`, because its phrasing dodges the stub's word lists. Keep that example in mind. It is the difference between a metrics table that reads 1.000 everywhere and one that tells you something.

### Checks so far

In [ ]:
check(
    "c01_config",
    "config is sane",
    lambda: (
        close_to(1, 1) if BACKEND_MODE in ("offline", "lmstudio", "ollama") else (_ for _ in ()).throw(AssertionError("bad mode")),
    ),
)


def _c02():
    assert len(corpus_df) == 300, f"expected 300 docs, got {len(corpus_df)}"
    counts = corpus_df["doc_type"].value_counts().to_dict()
    expected = {
        "policy_faq": 55, "promotion": 54, "installation_guide": 52,
        "troubleshooting": 50, "customer_review": 48, "product_guide": 41,
    }
    assert counts == expected, f"doc_type counts differ: {counts}"


check("c02_corpus", "corpus is the expected deterministic build", _c02)


def _c03():
    assert len(eval_df) == 60, f"expected 60 eval rows, got {len(eval_df)}"
    counts = eval_df["task_type"].value_counts().to_dict()
    assert counts == {"sentiment": 25, "faq": 20, "summarization": 15}, counts
    sent = eval_df[eval_df["task_type"] == "sentiment"]["sentiment_label"].value_counts().to_dict()
    assert sent == {"negative": 9, "neutral": 9, "positive": 7}, sent


check("c03_evalset", "evaluation dataset is the expected deterministic build", _c03)


def _c04():
    require_offline()
    out = run_cordwell_app(
        "What is the return policy for power tools at Cordwell?", "faq"
    )
    assert isinstance(out, dict) and "answer" in out and "contexts" in out
    assert out["answer"].startswith("The Cordwell return policy for power tools"), out["answer"][:80]
    assert len(out["contexts"]) == 4


check("c04_app_smoke", "app answers deterministically in offline mode", _c04)

## 4. Traditional metrics

Four small functions, four tasks. Together they are the scoring core of the harness. The shared tokenizer below is GIVEN so every function splits text the same way: lowercase, keep runs of letters and digits, drop everything else.

Two library traps to respect, both verified against the pinned versions:

- `rouge_scorer.RougeScorer.score` takes the **target (reference) first and the prediction second**. That is backwards from how most people think about it and backwards from sacrebleu. Swapping them does not error. It silently computes a different number.
- `sacrebleu.corpus_bleu(...).score` is on a **0 to 100 scale**. We store all metrics on 0 to 1, so divide by 100.

In [ ]:
import sacrebleu
from rouge_score import rouge_scorer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

_WORD_RE = re.compile(r"[a-z0-9]+")


def tokenize(text: str) -> list:
    """Lowercase the text and return runs of letters and digits."""
    return _WORD_RE.findall(text.lower())


print(tokenize("Returns within 90 days, with proof-of-purchase!"))

### Task 1: token-level F1

Token F1 asks: of the vocabulary the prediction used, how much overlaps the reference, and of the vocabulary the reference used, how much did the prediction cover? It is the harmonic mean of those two rates over **unique** tokens. Order and repetition are ignored, which is exactly why we also want BLEU and ROUGE next.

**Worked target output**

```text
>>> token_f1(
...     "unopened laminate flooring can be returned within 90 days with a receipt",
...     "most unopened laminate flooring purchases can be returned within the posted return window with proof of purchase",
... )
0.5517241379310345
```

The prediction has 12 unique tokens, the reference 17, and they share 8. Precision 8 over 12, recall 8 over 17, harmonic mean 0.5517.

In [ ]:
def token_f1(prediction: str, reference: str) -> float:
    """Token-level F1 over unique tokens.

    Contract:
    - Tokenize both strings with tokenize() and treat each as a SET of
      unique tokens.
    - If either set is empty, or the overlap is empty, return 0.0.
    - precision = overlap size over prediction set size
    - recall = overlap size over reference set size
    - Return the harmonic mean: 2 * precision * recall / (precision + recall).
    """
    # TODO: implement per the contract above
    raise NotImplementedError

In [ ]:
def _c05():
    close_to(
        token_f1(
            "unopened laminate flooring can be returned within 90 days with a receipt",
            "most unopened laminate flooring purchases can be returned within the posted return window with proof of purchase",
        ),
        0.5517241379310345,
    )
    close_to(token_f1("", "anything"), 0.0)
    close_to(token_f1("alpha beta", "gamma delta"), 0.0)
    close_to(token_f1("same words here", "same words here"), 1.0)


check("c05_token_f1", "token_f1 matches the worked target and edge cases", _c05)

### Task 2: the text metrics bundle

One function that scores a whole batch of predictions against references and returns three numbers: corpus BLEU, mean ROUGE-L F-measure, and mean token F1. This is the function the harness will call for the FAQ and summarization tasks.

**Worked target output**

```text
>>> compute_text_metrics(
...     ["the return window is 90 days for unopened items", "keep your receipt for any refund"],
...     ["unopened items can be returned within 90 days", "refunds require the original receipt"],
... )
{'bleu': 0.08970363701148239, 'rougeL_f': 0.26737967914438504, 'token_f1': 0.3262032085561497}
```

Notice how low BLEU is on answers a human would call half-decent. BLEU counts matching 1-gram to 4-gram sequences, and free-form paraphrase shares vocabulary but not word order. This gap is the running theme of the week.

In [ ]:
def compute_text_metrics(predictions: list, references: list) -> dict:
    """Score a batch of text predictions against references.

    Contract:
    - bleu: sacrebleu.corpus_bleu(predictions, [references]).score,
      divided by 100.0 so it lands on the 0 to 1 scale. Note the second
      argument is a LIST OF REFERENCE LISTS, hence the extra brackets.
    - rougeL_f: build one rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True),
      score each pair, take the rougeL fmeasure, and average with np.mean.
      CAUTION: scorer.score takes the reference FIRST and the prediction
      SECOND. Swapping them computes a different number without erroring.
    - token_f1: average of token_f1(pred, ref) over the pairs.
    - Return {"bleu": ..., "rougeL_f": ..., "token_f1": ...} with plain
      Python floats.
    """
    # TODO: implement per the contract above
    raise NotImplementedError

In [ ]:
def _c06():
    result = compute_text_metrics(
        ["the return window is 90 days for unopened items", "keep your receipt for any refund"],
        ["unopened items can be returned within 90 days", "refunds require the original receipt"],
    )
    assert set(result) == {"bleu", "rougeL_f", "token_f1"}, f"keys: {sorted(result)}"
    close_to(result["bleu"], 0.08970363701148239, 1e-9)
    close_to(result["rougeL_f"], 0.26737967914438504, 1e-9)
    close_to(result["token_f1"], 0.3262032085561497, 1e-9)


check("c06_text_metrics", "compute_text_metrics matches the worked target", _c06)

### Tasks 3 and 4: sentiment scoring

Real models do not reply with exactly one clean word. They reply with things like `Sentiment: NEGATIVE.` or a full sentence. Task 3 normalizes raw output into the three-label space. Task 4 computes classification metrics on the normalized labels, reusing the exact sklearn calls from Lab 1.2 earlier this week.

**Worked target output**

```text
>>> normalize_sentiment_label("Sentiment: NEGATIVE.")
'negative'
>>> normalize_sentiment_label("The review reads as mixed.")
'neutral'

>>> compute_sentiment_metrics(
...     ["positive", "neutral", "negative", "neutral", "positive", "neutral"],
...     ["positive", "neutral", "negative", "negative", "positive", "positive"],
... )
{'accuracy': 0.6666666666666666, 'precision_macro': 0.7777777777777777,
 'recall_macro': 0.7222222222222222, 'f1_macro': 0.6555555555555556}
```

In [ ]:
VALID_LABELS = ("positive", "negative", "neutral")


def normalize_sentiment_label(raw_answer: str) -> str:
    """Map a raw model reply to one of VALID_LABELS.

    Contract:
    - Lowercase the reply, then return the FIRST label from VALID_LABELS
      found as a substring, checking in VALID_LABELS order.
    - If none of the labels appear, return "neutral" as the fallback.
    """
    # TODO: implement per the contract above
    raise NotImplementedError

In [ ]:
def compute_sentiment_metrics(predictions: list, references: list) -> dict:
    """Classification metrics for normalized sentiment labels.

    Contract:
    - accuracy: sklearn accuracy_score(references, predictions)
    - precision_macro, recall_macro, f1_macro: from
      precision_recall_fscore_support with average="macro" and
      zero_division=0 (a class the model never predicts should score 0,
      not raise a warning).
    - Return {"accuracy": ..., "precision_macro": ..., "recall_macro": ...,
      "f1_macro": ...} with plain Python floats.
    """
    # TODO: implement per the contract above
    raise NotImplementedError

In [ ]:
def _c07():
    assert normalize_sentiment_label("Sentiment: NEGATIVE.") == "negative"
    assert normalize_sentiment_label("It was Positive overall!") == "positive"
    assert normalize_sentiment_label("The review reads as mixed.") == "neutral"


check("c07_normalize", "normalize_sentiment_label matches the worked target", _c07)


def _c08():
    result = compute_sentiment_metrics(
        ["positive", "neutral", "negative", "neutral", "positive", "neutral"],
        ["positive", "neutral", "negative", "negative", "positive", "positive"],
    )
    close_to(result["accuracy"], 0.6666666666666666, 1e-9)
    close_to(result["precision_macro"], 0.7777777777777777, 1e-9)
    close_to(result["recall_macro"], 0.7222222222222222, 1e-9)
    close_to(result["f1_macro"], 0.6555555555555556, 1e-9)


check("c08_sentiment_metrics", "compute_sentiment_metrics matches the worked target", _c08)

## 5. Task 5: the evaluation harness

Everything so far scores a batch you hand it. The harness closes the loop: point it at a variant of the app and it runs the whole evaluation set, routes each task type to the right metric function, and returns one tidy result. An evaluation harness is unit testing for model behavior: same discipline, fuzzier assertions.

**Worked target output** (offline mode, baseline variant)

```text
>>> result = evaluate_variant("prompt_baseline_v1", prompt_style="baseline", top_k=4)
>>> for key in sorted(result["metrics"]):
...     print(key, round(result["metrics"][key], 4))
faq_bleu 0.3026
faq_rougeL_f 0.4029
faq_token_f1 0.5213
sentiment_accuracy 0.84
sentiment_f1_macro 0.8471
sentiment_precision_macro 0.8974
sentiment_recall_macro 0.8413
summarization_bleu 0.2788
summarization_rougeL_f 0.5295
summarization_token_f1 0.5312
>>> len(result["predictions"])
60
```

Read that table for a second before implementing. Sentiment accuracy is 0.84, not 1.0, because of the review phrasings the stub cannot see. FAQ BLEU is 0.30 on answers that are literally copied from the correct policy documents. The numbers are already telling the story this week is about.

In [ ]:
def evaluate_variant(
    variant_name: str,
    prompt_style: str = "baseline",
    top_k: int = 4,
    n_faq: int = 20,
    n_summ: int = 15,
    n_sent: int = 25,
) -> dict:
    """Run one app variant over the evaluation set and score it.

    Contract:
    - For each task type in the order faq, summarization, sentiment, take
      the first n rows of that task type from eval_df with .head(limit),
      where the limits are n_faq, n_summ, n_sent.
    - For each row call run_cordwell_app(row["input_text"], task_type,
      top_k=top_k, prompt_style=prompt_style).
    - For sentiment rows the prediction is
      normalize_sentiment_label(answer) and the reference is
      row["sentiment_label"]. For the other tasks the prediction is the
      raw answer and the reference is row["reference_text"].
    - Also append one dict per row to a running prediction_rows list with
      keys: variant_name, task_type, input_text, reference, prediction.
    - After each task's loop, score it: compute_sentiment_metrics for
      sentiment, compute_text_metrics otherwise. Copy every metric into
      one flat dict with the task type as prefix, for example
      "faq_bleu" and "sentiment_accuracy". Skip a task whose limit is 0.
    - Return {"variant_name": variant_name, "metrics": metrics,
      "predictions": pd.DataFrame(prediction_rows)}.
    """
    # TODO: implement per the contract above
    raise NotImplementedError

In [ ]:
# Run the baseline variant through the harness (guarded so a fresh
# Run All does not crash before Task 5 is implemented).
baseline_result = None
try:
    baseline_result = evaluate_variant("prompt_baseline_v1", prompt_style="baseline", top_k=4)
    for key in sorted(baseline_result["metrics"]):
        print(f"{key:32s} {baseline_result['metrics'][key]:.4f}")
    print(f"prediction rows: {len(baseline_result['predictions'])}")
except NotImplementedError:
    print("Task 5 not implemented yet; skipping the baseline run.")


def _c09():
    require_offline()
    if baseline_result is None:
        raise NotImplementedError("finish Task 5 and re-run this cell")
    m = baseline_result["metrics"]
    expected = {
        "faq_bleu": 0.3026316154806789,
        "faq_rougeL_f": 0.4029411764705883,
        "faq_token_f1": 0.521264367816092,
        "summarization_bleu": 0.2788364947127007,
        "summarization_rougeL_f": 0.5294794794794794,
        "summarization_token_f1": 0.53125,
        "sentiment_accuracy": 0.84,
        "sentiment_precision_macro": 0.8974358974358975,
        "sentiment_recall_macro": 0.8412698412698413,
        "sentiment_f1_macro": 0.8470862470862471,
    }
    assert set(m) == set(expected), f"metric keys differ: {sorted(m)}"
    for key, value in expected.items():
        close_to(m[key], value, 1e-6)
    assert len(baseline_result["predictions"]) == 60
    expected_cols = {"variant_name", "task_type", "input_text", "reference", "prediction"}
    assert set(baseline_result["predictions"].columns) == expected_cols


check("c09_harness", "evaluate_variant reproduces the locked baseline metrics", _c09)

## 6. Task 6: log the experiment to MLflow

A metrics printout dies with the kernel. MLflow gives every experiment a permanent record with three kinds of content, and your function will log all three:

- **Params** are the knobs you chose: variant name, prompt style, top_k, backend, model. Params answer "what did we run?"
- **Metrics** are the numbers that came out. Metrics answer "how did it do?"
- **Artifacts** are files attached to the run. The per-example predictions CSV is the difference between "the score dropped" and "here are the exact examples that got worse."

The setup cell below is GIVEN. It points MLflow at the tracking URI from Section 0 and selects the experiment, creating it on first use.

**Worked target output** (offline mode)

```text
>>> run_id = log_variant_to_mlflow(baseline_result, prompt_style="baseline", top_k=4)
>>> print(run_id)
a run id like 2ca9650ed770476f8e4e32724670ce82
```

and in the MLflow UI (or via `MlflowClient`), a run named `prompt_baseline_v1` holding 5 params, the 10 harness metrics, and `predictions/prompt_baseline_v1_predictions.csv`.

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"MLflow experiment: {EXPERIMENT_NAME}")
print(f"Tracking URI     : {mlflow.get_tracking_uri()}")

In [ ]:
def log_variant_to_mlflow(result: dict, prompt_style: str, top_k: int) -> str:
    """Log one evaluate_variant result as an MLflow run.

    Contract:
    - Open a run with mlflow.start_run(run_name=result["variant_name"])
      as a context manager.
    - Log five params: variant_name, prompt_style, top_k, backend_mode
      (the BACKEND_MODE global), model_name (the MODEL_NAME global).
    - Log every entry of result["metrics"] with mlflow.log_metric.
    - Write result["predictions"] to a CSV named
      "<variant_name>_predictions.csv" inside LAB_DIR (index=False),
      then mlflow.log_artifact it under artifact_path="predictions".
    - Return the run id from the run context object (run.info.run_id).
    """
    # TODO: implement per the contract above
    raise NotImplementedError

In [ ]:
baseline_run_id = None
try:
    if baseline_result is None:
        print("Need Task 5 first; skipping the MLflow log.")
    else:
        baseline_run_id = log_variant_to_mlflow(baseline_result, prompt_style="baseline", top_k=4)
        print(f"Logged MLflow run: {baseline_run_id}")
except NotImplementedError:
    print("Task 6 not implemented yet; skipping the MLflow log.")


def _c10():
    require_offline()
    if baseline_run_id is None:
        raise NotImplementedError("finish Tasks 5 and 6 and re-run this cell")
    client = MlflowClient()
    experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
    assert experiment is not None, "experiment missing"
    runs = client.search_runs(
        [experiment.experiment_id],
        filter_string="tags.mlflow.runName = 'prompt_baseline_v1'",
        order_by=["attributes.start_time DESC"],
        max_results=1,
    )
    assert runs, "no run named prompt_baseline_v1"
    run = runs[0]
    for param in ("variant_name", "prompt_style", "top_k", "backend_mode", "model_name"):
        assert param in run.data.params, f"missing param {param}"
    close_to(run.data.metrics["faq_bleu"], 0.3026316154806789, 1e-6)
    close_to(run.data.metrics["sentiment_accuracy"], 0.84, 1e-6)
    artifacts = [a.path for a in client.list_artifacts(run.info.run_id, "predictions")]
    assert any(p.endswith("prompt_baseline_v1_predictions.csv") for p in artifacts), artifacts


check("c10_mlflow", "MLflow run holds the params, metrics, and predictions artifact", _c10)

### Look at what you just logged

- **Docker server mode:** open `http://localhost:5000`, select the `week6_lab51_cordwell_eval` experiment, and click into the run.
- **Local file mode:** in a terminal inside the lab folder (venv active), run `mlflow ui --backend-store-uri sqlite:///mlflow_local.db --port 5001` and open `http://localhost:5001`.

Find your five params, the ten metrics, and download the predictions CSV from the artifacts tab. This run now outlives your kernel, your laptop battery, and your memory of what `top_k` was set to.

## 7. Structured evaluation with TruLens

BLEU and ROUGE compare an answer to a reference. They cannot see the retrieved context, so they cannot tell you the one thing a RAG team cares about most: **did the model make it up?** Structured evaluation scores each request against the RAG triad instead:

| Metric | Question it answers | Inputs |
|---|---|---|
| Context relevance | Did retrieval fetch material related to the question? | question, each retrieved context |
| Groundedness | Is the answer supported by the retrieved context? | answer, all contexts combined |
| Answer relevance | Does the answer address the question? | question, answer |

### How TruLens sees your app

TruLens instruments the app with **spans**, the same concept as OpenTelemetry tracing you may know from microservices. Decorating a method tells TruLens what role it plays:

- `retrieve` is a RETRIEVAL span. We map its `query` argument to the span's query text and its return value to the retrieved contexts.
- `generate` is a GENERATION span.
- `answer_question` is the RECORD_ROOT span, the top of the trace for one request. Its input and output are the user question and the final answer.

A **Metric** is just a Python function returning a score between 0 and 1, plus **Selectors** that tell TruLens which span attributes to feed into which function parameter. In production the function usually calls a judge LLM. In this lab you will write cheap deterministic **lexical judges** based on token overlap, for three reasons: they run instantly with no server, they make the check numbers exact, and building one yourself demystifies what any judge, LLM or not, actually is: a function from evidence to a score. The stretch section swaps in TruLens stock LLM judges on a live backend.

### One rule that will save you an hour

TruLens computes metric scores **asynchronously in background threads**. Verified behavior on trulens 2.12.0: if you start recording a second `TruApp` while the first one still has pending evaluations, those pending evaluations are dropped and their scores stay NaN forever. The given `wait_for_trulens` helper polls until every score has landed. **Always wait between app versions.** The same applies to reusing `Metric` objects across apps, which is why `build_trulens_metrics` constructs fresh ones each time it is called.

In [ ]:
from trulens.core import TruSession, Metric
from trulens.core.otel.instrument import instrument
from trulens.otel.semconv.trace import SpanAttributes
from trulens.core.feedback.selector import Selector
from trulens.apps.app import TruApp

session = TruSession()
session.reset_database()  # start clean so re-runs do not mix with old records


class CordwellRAG:
    """The same app as Section 3, wrapped in a class and instrumented
    with TruLens spans. Fixed to the faq task for structured evaluation."""

    def __init__(self, top_k: int = 4, prompt_style: str = "baseline"):
        self.top_k = top_k
        self.prompt_style = prompt_style

    @instrument(
        span_type=SpanAttributes.SpanType.RETRIEVAL,
        attributes={
            SpanAttributes.RETRIEVAL.QUERY_TEXT: "query",
            SpanAttributes.RETRIEVAL.RETRIEVED_CONTEXTS: "return",
        },
    )
    def retrieve(self, query: str) -> list:
        return retrieve_contexts(query, top_k=self.top_k)

    @instrument(span_type=SpanAttributes.SpanType.GENERATION)
    def generate(self, query: str, contexts: list) -> str:
        messages = build_messages(query, contexts, "faq", self.prompt_style)
        return call_llm(messages)

    @instrument(span_type=SpanAttributes.SpanType.RECORD_ROOT)
    def answer_question(self, query: str) -> str:
        contexts = self.retrieve(query=query)
        return self.generate(query=query, contexts=contexts)


TRULENS_QUERIES = [
    "What is the return policy for laminate flooring at Cordwell?",
    "How long is the warranty on vinyl plank flooring?",
    "What should I check before installing ceramic tile in a bathroom?",
    "Can I return custom-tinted interior paint if I do not like the color?",
    "How should I clean LED lighting fixtures from Cordwell?",
    "Do you ever run promotions on power tools?",
]

METRIC_NAMES = [
    "Context Relevance (lexical)",
    "Groundedness (lexical)",
    "Answer Relevance (lexical)",
]


def wait_for_trulens(expected_records: int, metric_names: list, timeout_seconds: int = 300):
    """Poll the TruLens session until every record has every metric score.

    Metric scores are computed in background threads. Always wait for
    completion before starting another TruApp, or pending evaluations
    can be dropped and stay NaN forever (observed on trulens 2.12.0).
    """
    session.force_flush()
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        records_df, feedback_cols = session.get_records_and_feedback()
        ready = (
            len(records_df) >= expected_records
            and all(name in feedback_cols for name in metric_names)
            and records_df[metric_names].notna().all(axis=1).all()
        )
        if ready:
            return records_df, feedback_cols
        time.sleep(2)
    print(f"Timed out after {timeout_seconds}s; some scores may still be NaN.")
    return session.get_records_and_feedback()


STOPWORDS = {
    "the", "a", "an", "and", "or", "for", "with", "what", "how", "is", "are",
    "do", "does", "can", "i", "my", "at", "on", "in", "of", "to", "before",
}


def content_tokens(text: str) -> set:
    """Tokenize and drop stopwords, keeping only meaning-bearing tokens."""
    return {t for t in tokenize(text) if t not in STOPWORDS}


def lexical_context_relevance(question: str, context: str) -> float:
    """GIVEN worked example of a lexical judge. Fraction of the question's
    content tokens that appear in one retrieved context."""
    question_tokens = content_tokens(question)
    if not question_tokens:
        return 0.0
    context_tokens = content_tokens(context)
    return len(question_tokens & context_tokens) / len(question_tokens)


print("TruLens session ready. Instrumented app and helpers defined.")

### Task 7: write the offline judges

Two functions matching the shapes of the remaining triad metrics. Model them on the GIVEN `lexical_context_relevance` above: normalize to content tokens, intersect, divide. What changes is **which side supplies the denominator**, and that choice is the meaning of the metric:

- Groundedness divides by the **statement's** tokens: how much of what the answer said is backed by the source?
- Answer relevance divides by the **prompt's** tokens: how much of what was asked does the answer engage with?

**Worked target output**

```text
>>> lexical_groundedness(
...     "Unopened tile can be returned within 60 days.",
...     ["Most unopened ceramic tile purchases can be returned within 60 days with proof of purchase."],
... )
1.0
>>> lexical_answer_relevance(
...     "What is the return policy for ceramic tile?",
...     "Most unopened ceramic tile purchases can be returned within 60 days.",
... )
0.5
```

That 0.5 is worth a pause: the answer clearly addresses the question, but the judge only credits `ceramic` and `tile` because `return` and `returned` do not match as strings. Lexical judges are cheap and transparent and exactly this crude. Hold that thought for Stretch 3, where an LLM judge replaces string overlap with understanding.

In [ ]:
def lexical_groundedness(statement: str, source) -> float:
    """Fraction of the statement's content tokens found in the source.

    Contract:
    - source may be a single string OR a list of strings. If it is a
      list or tuple, join it into one string with spaces first.
    - Use content_tokens() on both sides.
    - If the statement has no content tokens, return 0.0.
    - Return overlap size divided by the STATEMENT's token count.
    """
    # TODO: implement per the contract above
    raise NotImplementedError


def lexical_answer_relevance(prompt: str, response: str) -> float:
    """Fraction of the prompt's content tokens found in the response.

    Contract:
    - Use content_tokens() on both sides.
    - If the prompt has no content tokens, return 0.0.
    - Return overlap size divided by the PROMPT's token count.
    """
    # TODO: implement per the contract above
    raise NotImplementedError

In [ ]:
def _c11():
    close_to(
        lexical_context_relevance(
            "What is the return policy for ceramic tile?",
            "The Cordwell return policy for ceramic tile balances flexibility with product quality.",
        ),
        1.0,
    )
    close_to(
        lexical_groundedness(
            "Unopened tile can be returned within 60 days.",
            ["Most unopened ceramic tile purchases can be returned within 60 days with proof of purchase."],
        ),
        1.0,
    )
    close_to(
        lexical_groundedness("something entirely fabricated here", ["unrelated source text"]),
        0.0,
    )
    close_to(
        lexical_answer_relevance(
            "What is the return policy for ceramic tile?",
            "Most unopened ceramic tile purchases can be returned within 60 days.",
        ),
        0.5,
    )


check("c11_judges", "lexical judges match the worked targets", _c11)

### Task 8: wire the judges into TruLens Metrics

A judge function alone cannot score anything, because TruLens does not know which pieces of the trace to feed it. That mapping is the job of `Selector`: each `.on({...})` call binds one function parameter to one span attribute. The factory below has the context relevance metric **fully wired as the worked example**. Study its three moves, then wire the other two the same way:

1. `Metric(implementation=fn, name="...")` names the metric. `Metric` is the current API on trulens 2.12; the `Feedback` class you may see in older tutorials is deprecated and prints a warning.
2. Each `.on({"param": Selector(...)})` maps a parameter name from the function signature to a span attribute. Context relevance reads both of its inputs from the RETRIEVAL span.
3. `collect_list` controls fan-out on list-valued attributes. `collect_list=False` calls the function once per retrieved context, which is why context relevance ends with `.aggregate(np.mean)`. `collect_list=True` hands the whole list to a single call, which is what groundedness wants: the answer judged against all evidence at once.

Your two metrics read from these span attributes:

| Metric name | Parameter | Selector span and attribute |
|---|---|---|
| `Groundedness (lexical)` | `statement` | RECORD_ROOT span, `RECORD_ROOT.OUTPUT` |
| `Groundedness (lexical)` | `source` | RETRIEVAL span, `RETRIEVAL.RETRIEVED_CONTEXTS`, `collect_list=True` |
| `Answer Relevance (lexical)` | `prompt` | RECORD_ROOT span, `RECORD_ROOT.INPUT` |
| `Answer Relevance (lexical)` | `response` | RECORD_ROOT span, `RECORD_ROOT.OUTPUT` |

The parameter names in the Selector dicts must match the function signatures from Task 7 exactly. Neither of your metrics needs `.aggregate`.

In [ ]:
def build_trulens_metrics() -> list:
    """Build fresh Metric objects wired to the instrumented span attributes.

    Returns fresh objects each call on purpose: reusing Metric objects
    across TruApps is fragile on trulens 2.12 (see the rule above).
    """
    m_context_relevance = (
        Metric(implementation=lexical_context_relevance, name="Context Relevance (lexical)")
        .on(
            {
                "question": Selector(
                    span_type=SpanAttributes.SpanType.RETRIEVAL,
                    span_attribute=SpanAttributes.RETRIEVAL.QUERY_TEXT,
                )
            }
        )
        .on(
            {
                "context": Selector(
                    span_type=SpanAttributes.SpanType.RETRIEVAL,
                    span_attribute=SpanAttributes.RETRIEVAL.RETRIEVED_CONTEXTS,
                    collect_list=False,
                )
            }
        )
        .aggregate(np.mean)
    )

    # TODO: build m_groundedness. Name it "Groundedness (lexical)", use
    # lexical_groundedness as the implementation, and wire "statement"
    # and "source" per the table above.
    m_groundedness = None

    # TODO: build m_answer_relevance. Name it "Answer Relevance (lexical)",
    # use lexical_answer_relevance as the implementation, and wire
    # "prompt" and "response" per the table above.
    m_answer_relevance = None

    if m_groundedness is None or m_answer_relevance is None:
        raise NotImplementedError("Task 8: wire the two remaining metrics")

    return [m_context_relevance, m_groundedness, m_answer_relevance]

In [ ]:
def _c12():
    metrics = build_trulens_metrics()
    assert len(metrics) == 3, f"expected 3 metrics, got {len(metrics)}"
    names = [m.name for m in metrics]
    assert names == METRIC_NAMES, f"names differ: {names}"
    assert all(isinstance(m, Metric) for m in metrics)


check("c12_metric_wiring", "build_trulens_metrics returns the three named Metrics", _c12)

In [ ]:
def run_trulens_eval(app_version: str, top_k: int, prompt_style: str, expected_total: int):
    """Record one app version over TRULENS_QUERIES and wait for scores."""
    rag_variant = CordwellRAG(top_k=top_k, prompt_style=prompt_style)
    tru_variant = TruApp(
        rag_variant,
        app_name="CordwellRAG",
        app_version=app_version,
        feedbacks=build_trulens_metrics(),
    )
    with tru_variant:
        for query in TRULENS_QUERIES:
            rag_variant.answer_question(query)
    return wait_for_trulens(expected_total, METRIC_NAMES)


# Guarded runner: only fires once Tasks 7 and 8 are implemented.
trulens_ready = False
try:
    lexical_groundedness("probe statement", ["probe source"])
    lexical_answer_relevance("probe prompt", "probe response")
    build_trulens_metrics()
    trulens_ready = True
except NotImplementedError:
    print("Tasks 7 and 8 not finished yet; skipping the TruLens run.")

records_df = None
if trulens_ready:
    records_df, feedback_cols = run_trulens_eval(
        "prompt_baseline_v1", top_k=4, prompt_style="baseline",
        expected_total=len(TRULENS_QUERIES),
    )
    display(records_df[["app_version", "input"] + METRIC_NAMES].round(3))
    leaderboard = session.get_leaderboard().reset_index()
    display(leaderboard.round(4))


def _c13():
    require_offline()
    if records_df is None:
        raise NotImplementedError("finish Tasks 7 and 8 and re-run this cell")
    assert len(records_df) >= len(TRULENS_QUERIES)
    lb = session.get_leaderboard().reset_index()
    row = lb[lb["app_version"] == "prompt_baseline_v1"].iloc[0]
    close_to(float(row["Groundedness (lexical)"]), 1.0, 1e-6)
    close_to(float(row["Context Relevance (lexical)"]), 0.6119047619047618, 1e-6)
    close_to(float(row["Answer Relevance (lexical)"]), 0.5047619047619047, 1e-6)


check("c13_trulens", "TruLens leaderboard matches the locked baseline scores", _c13)

### Read the results like a reviewer

Three observations to make on the per-record table before moving on:

1. **Groundedness is a flat 1.000.** Of course it is: the stub copies sentences straight out of the retrieved context, and the strictest possible groundedness judge cannot fault a photocopy. A perfect score is only meaningful once you know what the app does.
2. **Answer relevance disagrees with groundedness.** The promotions question scored 0.333: the stub returned a perfectly grounded chunk of the wrong document. One request, two metrics, two different verdicts. That is why the triad has three legs.
3. **Context relevance varies from 0.333 to 1.000.** When it is low, blame retrieval, not generation. The triad localizes failure: retrieval problems show up in context relevance, generation problems show up in groundedness.

And a caveat to say out loud: these lexical judges reward copying and cannot credit a paraphrase. When Stretch 3 puts an LLM judge on the same records, groundedness will stop being a photocopier detector and start being a fact checker.

## 8. Task 9: join TruLens scores into MLflow

Right now the traditional metrics live in MLflow and the structured scores live in TruLens. A reviewer deciding which variant ships should not need two dashboards. Your last function copies a version's leaderboard scores onto its MLflow run as `trulens_*` metrics.

The clean way to do this: `session.get_leaderboard()` comes back indexed by app name and version, so `.reset_index()` turns those into ordinary columns you can filter on. On the MLflow side, `MlflowClient().search_runs` with a `runName` tag filter finds the run; sort by start time descending and take one, so re-running the notebook joins onto the newest run rather than a stale duplicate.

**Worked target output** (offline mode)

```text
>>> attach_trulens_metrics_to_mlflow("prompt_baseline_v1", "prompt_baseline_v1")
{'trulens_context_relevance_lexical': 0.6119047619047618,
 'trulens_groundedness_lexical': 1.0,
 'trulens_answer_relevance_lexical': 0.5047619047619047}
```

In [ ]:
def attach_trulens_metrics_to_mlflow(app_version: str, run_name: str) -> dict:
    """Copy one app version's TruLens leaderboard scores onto an MLflow run.

    Contract:
    - Get the leaderboard with session.get_leaderboard().reset_index()
      and filter rows where the app_version column equals app_version.
      Raise ValueError if no row matches. Take the first matching row.
    - Find the MLflow run: MlflowClient().search_runs on the experiment
      (get it by EXPERIMENT_NAME; raise ValueError if missing) with
      filter_string "tags.mlflow.runName = '<run_name>'",
      order_by=["attributes.start_time DESC"], max_results=1.
      Raise ValueError if no run matches.
    - For each name in METRIC_NAMES present and non-null in the row:
      build the key as "trulens_" plus the name lowercased with every
      run of characters outside a to z and 0 to 9 replaced by one
      underscore, with leading and trailing underscores stripped
      (re.sub is your friend). Log it with client.log_metric(run_id,
      key, float(value)).
    - Return a dict of everything logged, key to value.
    """
    # TODO: implement per the contract above
    raise NotImplementedError

In [ ]:
joined = None
try:
    if records_df is None or baseline_run_id is None:
        print("Need the TruLens run and the MLflow run first; skipping the join.")
    else:
        joined = attach_trulens_metrics_to_mlflow("prompt_baseline_v1", "prompt_baseline_v1")
        print("Attached to MLflow:")
        for key, value in joined.items():
            print(f"  {key}: {value:.4f}")
except NotImplementedError:
    print("Task 9 not implemented yet; skipping the join.")


def _c14():
    require_offline()
    if joined is None:
        raise NotImplementedError("finish Task 9 and re-run this cell")
    client = MlflowClient()
    experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
    runs = client.search_runs(
        [experiment.experiment_id],
        filter_string="tags.mlflow.runName = 'prompt_baseline_v1'",
        order_by=["attributes.start_time DESC"],
        max_results=1,
    )
    metrics = runs[0].data.metrics
    close_to(metrics["trulens_groundedness_lexical"], 1.0, 1e-6)
    close_to(metrics["trulens_context_relevance_lexical"], 0.6119047619047618, 1e-6)
    close_to(metrics["trulens_answer_relevance_lexical"], 0.5047619047619047, 1e-6)
    close_to(metrics["faq_bleu"], 0.3026316154806789, 1e-6)


check("c14_join", "MLflow run now holds both metric families", _c14)

## 9. Stretch goals (optional)

Solutions to every stretch goal are in the instructor solution notebook, and `HINTS_DETAILED.md` covers them at the same depth as the core tasks.

### Stretch 1: run the A-B test end to end

Variant B exists in the code already: `prompt_style="strong_grounding"` with `top_k=2`. Push it through the entire chain you just built:

1. `evaluate_variant("prompt_grounded_v2", prompt_style="strong_grounding", top_k=2)` and log it to MLflow.
2. Run the TruLens evaluation for an app_version `prompt_grounded_v2` with `top_k=2` and `strong_grounding`, waiting for completion.
3. Attach its TruLens scores to its MLflow run.
4. Compare the two runs. In offline mode you should find that variant B scores **higher on answer relevance but lower on groundedness**, and the reason is one specific sentence the stub appends. Find it, explain the trade, and decide which variant you would ship.

### Stretch 2: invent a metric, measure a behavior

The triad is not a closed list. Any behavior you can detect in a span is a metric. Write a `hedge_detector(response)` function that returns 1.0 when the answer hedges (contains phrases like "do not know" or "check with an associate") and 0.0 otherwise, wrap it in a `Metric` named `Hedge Rate` on the RECORD_ROOT output, and run both prompt styles under new app versions. In offline mode the baseline style scores 0.0 and strong grounding scores 1.0, which turns a vague prompt-engineering intuition into a tracked number.

### Stretch 3: swap in real LLM judges (live backend required)

With LM Studio or Ollama running, TruLens stock judges replace your lexical ones. The instructor solution notebook has the full code in an appendix. The short version: build a provider with `TruOpenAI(model_engine=MODEL_NAME)` from `trulens.providers.openai`, then use `provider.context_relevance`, `provider.groundedness_measure_with_cot_reasons`, and `provider.relevance` as the metric implementations with the exact same Selectors you wired in Task 8. Compare the LLM judge scores against your lexical ones on the same six queries and note where they disagree and why.

## 10. Wrap-up

You built the full evaluation loop a production LLM team runs before every release:

- **Traditional metrics** scored free-form output against references and labels, and you handled both argument-order traps (`rouge_scorer.score(target, prediction)` and sacrebleu's 0 to 100 scale) that silently corrupt results when missed.
- **An evaluation harness** turned a labeled dataset plus an app variant into one metrics dictionary and a per-example prediction table.
- **MLflow** made every experiment reproducible and comparable: params, metrics, artifacts, side by side in one UI, whether backed by a local file or a Docker tracking server.
- **TruLens** scored what reference metrics cannot see: whether answers are grounded in retrieved evidence and relevant to the question. You wrote the judges yourself, so LLM-as-a-judge is no longer a black box, just a smarter implementation behind the same Metric interface.
- **The join** put structured scores and traditional scores on the same MLflow run, so one screen answers "which variant is better, and by what definition of better."

The final cell reruns every check. In offline mode a finished notebook shows all checks PASS with none in TODO or FAIL.

In [ ]:
run_summary()